In [1]:
import numpy as np
from scipy.optimize import minimize
import nltk
from collections import defaultdict, Counter
from scipy.sparse import csr_matrix
import joblib
import os
from joblib import Parallel, delayed
import json

In [36]:
# Function to extract adjectives
def extract_adjectives(document):
    words = nltk.word_tokenize(document)
    tagged_words = nltk.pos_tag(words)
    adjectives = [word for word, tag in tagged_words if tag.startswith('JJ')]
    return adjectives

# Vocabulary filtering based on word frequency
def filter_vocabulary(corpus, min_freq=5):
    counter = Counter()
    for doc in corpus:
        adjectives = extract_adjectives(doc)
        counter.update(adjectives)
    # Select tokens with frequency >= min_freq
    filtered_tokens = [token for token, freq in counter.items() if freq >= min_freq]
    return filtered_tokens

# Calculate sparse token distribution and pre-extract adjectives
def calculate_sparse_token_distribution(corpus, token_list):
    token_to_index = {token: idx for idx, token in enumerate(token_list)}
    n_docs = len(corpus)
    n_tokens = len(token_list)
    data = []
    row = []
    col = []
    extracted_adjectives = []
    for doc_idx, doc in enumerate(corpus):
        adjectives = set(extract_adjectives(doc))
        extracted_adjectives.append(adjectives)
        for adj in adjectives:
            if adj in token_to_index:
                token_idx = token_to_index[adj]
                data.append(1)
                row.append(doc_idx)
                col.append(token_idx)
    # Build sparse matrix
    X = csr_matrix((data, (row, col)), shape=(n_docs, n_tokens))
    # Calculate frequency of each word
    token_counts = np.array(X.sum(axis=0)).flatten()
    token_distribution = token_counts / n_docs
    return token_distribution, token_to_index, extracted_adjectives

# Calculate document log probability, use_frequency indicates whether to consider word frequency
def document_log_probability(extracted_adjectives, token_distribution, token_indices, use_frequency=False):
    # Get counts for each adjective if use_frequency, otherwise binary
    if use_frequency:
        adjective_counts = {adj: extracted_adjectives[adj] for adj in extracted_adjectives if adj in token_indices}
    else:
        unique_adjectives = list(set(extracted_adjectives))
        adjective_counts = {adj: 1 for adj in unique_adjectives if adj in token_indices}
    # Calculate log probability for each adjective
    indices = [token_indices[adj] for adj in adjective_counts]
    counts = np.array([adjective_counts[adj] for adj in adjective_counts])
    if len(indices) == 0:
        return np.log(1e-10)
    probs = token_distribution[indices]
    log_prob = np.sum(counts * np.log(probs + 1e-10))
    return log_prob

# Calculate overall log likelihood
def compute_log_likelihood(alpha, target_extracted_adjectives, human_token_distribution, ai_token_distribution, token_indices, use_frequency=False):
    log_likelihood = 0.0
    for adjectives in target_extracted_adjectives:
        P_x_log = document_log_probability(adjectives, human_token_distribution, token_indices,use_frequency)
        Q_x_log = document_log_probability(adjectives, ai_token_distribution, token_indices,use_frequency)
        # Use log-sum-exp trick for numerical stability
        max_log = max((1 - alpha) + P_x_log, alpha + Q_x_log)
        log_likelihood += max_log + np.log((1 - alpha) * np.exp(P_x_log - max_log) + alpha * np.exp(Q_x_log - max_log) + 1e-10)
    return -log_likelihood

# Optimization function
def estimate_alpha_optimized(target_extracted_adjectives, human_token_distribution, ai_token_distribution, token_indices, use_frequency=False):
    result = minimize(
        compute_log_likelihood,
        x0=np.array([0.5]),
        args=(target_extracted_adjectives, human_token_distribution, ai_token_distribution, token_indices, use_frequency),
        bounds=[(0, 1)],
        method='L-BFGS-B',
        options={'disp': True}
    )
    return result.x[0]

In [3]:
# Load or compute AI and Human corpus distributions
def load_or_compute_distributions(min_freq, filename='human_ai_distributions.joblib'):
    if os.path.exists(filename):
        print("Loading stored vocabulary distributions and indices...")
        data = joblib.load(filename)
        human_distribution = data['human_distribution']
        ai_distribution = data['ai_distribution']
        token_indices = data['token_indices']
    else:
        print("Computing vocabulary distributions and saving...")

        def separate_corpus(file_path):
            human_corpus = []
            ai_corpus = []
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            for entry in data:
                fake = entry.get('fake')
                text = entry.get('text', '').strip()
                if fake == 0:
                    human_corpus.append(text.lower())
                elif fake == 1:
                    ai_corpus.append(text.lower())
                else:
                    print(f"Warning: 'fake' value {fake} is not 0 or 1 for entry with URL {entry.get('url')}")
            return human_corpus, ai_corpus
        
        # JSON file paths (relative)
        json_file_path = 'train.json'
        human_corpus_1, ai_corpus_1 = separate_corpus(json_file_path)
        json_file_path = 'test.json'
        human_corpus_2, ai_corpus_2 = separate_corpus(json_file_path)
        json_file_path = 'val.json'
        human_corpus_3, ai_corpus_3 = separate_corpus(json_file_path)
        human_corpus = set(human_corpus_1 +human_corpus_2+human_corpus_3)
        ai_corpus = set(ai_corpus_1 +ai_corpus_2+ai_corpus_3)
        
        # Vocabulary filtering
        human_tokens = filter_vocabulary(human_corpus, min_freq=min_freq)
        ai_tokens = filter_vocabulary(ai_corpus, min_freq=min_freq)
        # Select common tokens
        common_tokens = list(set(human_tokens)|(set(ai_tokens)))
        print(f"Selected {len(common_tokens)} tokens out of possible.")
        human_distribution, token_indices, human_extracted_adjectives = calculate_sparse_token_distribution(human_corpus, common_tokens)
        ai_distribution, _, ai_extracted_adjectives = calculate_sparse_token_distribution(ai_corpus, common_tokens)
        data = {
            'human_distribution': human_distribution,
            'ai_distribution': ai_distribution,
            'token_indices': token_indices
        }
        joblib.dump(data, filename)
    return data['human_distribution'], data['ai_distribution'], data['token_indices']
    
human_distribution, ai_distribution, token_indices = load_or_compute_distributions(
min_freq=3, filename='human_ai_distributions.joblib'
)

Loading stored vocabulary distributions and indices...
